In [ ]:
import numpy as np
import gzip
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn

import seaborn as sns


# Cargamos el dataset MNIST y lo particionamos
En la siguiente celda cargaremos el dataset de MNIST y lo particionaremos según lo solicitado en 50,000 elementos para entrenamiento, 10,000 para validación y 10,000 para prueba. Para ello utilizaremos las funciones proporcionadas por el docente, load_mnist_dataset, get_labels y get_images, con las librerías path, gzip y numpy.

In [ ]:
def load_mnist_dataset(mnist_path):
    x_trainval = get_images(Path(mnist_path)/Path('train-images-idx3-ubyte.gz'))
    y_trainval = get_labels(Path(mnist_path)/Path('train-labels-idx1-ubyte.gz'))

    x_train = x_trainval[:50000]
    y_train = y_trainval[:50000]

    x_val = x_trainval[50000:]
    y_val = y_trainval[50000:]

    x_test = get_images(Path(mnist_path)/Path('t10k-images-idx3-ubyte.gz'))
    y_test = get_labels(Path(mnist_path)/Path('t10k-labels-idx1-ubyte.gz'))

    return x_train, y_train, x_val, y_val, x_test, y_test

def get_labels(path):
    with gzip.open(path, 'rb') as data:
        labels = data.read()[8:]
        return np.frombuffer(labels, dtype=np.uint8)

def get_images(path):
    with gzip.open(path, 'rb') as data:
        _ = int.from_bytes(data.read(4), 'big')
        num_images = int.from_bytes(data.read(4), 'big')
        rows = int.from_bytes(data.read(4), 'big')
        cols = int.from_bytes(data.read(4), 'big')
        images = data.read()
        return np.frombuffer(images, dtype=np.uint8).reshape((num_images, rows, cols))


In [ ]:
x_train, y_train, x_val, y_val, x_test, y_test = load_mnist_dataset('datasets/mnist')

A continuación, los volveremos matrices más tratables y las escalaremos de forma estándar, también con una función proporcionada en el notebook de referencia.

In [ ]:
x_train = x_train.copy().reshape(50000, -1).astype(np.float32)
y_train = y_train.copy().reshape(50000, 1)

x_val = x_val.copy().reshape(10000, -1).astype(np.float32)
y_val = y_val.copy().reshape(10000, 1)

x_test = x_test.copy().reshape(10000, -1).astype(np.float32)
y_test = y_test.copy().reshape(10000, 1)

def scale(x_mean, x_std, x_data):
    return (x_data - x_mean) / x_std

x_mean = x_train.mean()
x_std = x_train.std()

x_train = scale(x_mean, x_std, x_train)
x_val = scale(x_mean, x_std, x_val)
x_test = scale(x_mean, x_std, x_test)

Como paso final en el tratamiento de nuestros datos, convertiremos estos arreglos de NumPy a tensores de PyTorch con el tipo de dato adecuado, siendo los predictores en formato de punto flotante 32 bits y las etiquetas como un dato de tipo entero largo. Esto aplica para nuestros tensores de entrenamiento, validación y de prueba. Asimismo, crearemos el objeto TensorDataset a partir de nuestros datos de entrenamiento y finalmente crearemos el DataLoader a partir de nuestro dataset con un tamaño de batch de 32 y los sortearemos para evitar sesgar nuestro entrenamiento por mero ordenamiento de los datos.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.squeeze(), dtype=torch.long)
x_val_tensor = torch.tensor(x_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.squeeze(), dtype=torch.long)
x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.squeeze(), dtype=torch.long)

dataset = TensorDataset(x_train_tensor, y_train_tensor)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Red FeedForwardNN

Definimos una clase FeedForwardNN apartir de la cual crearemos los distintos tipos de modelos.

In [ ]:
class FeedForwardNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden = nn.Linear(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

Crearemos tres modelos que inicialmente diferirán en el tamaño de la capa oculta, teniendo el primero 32 neuronas, el segundo 250 y el tercero 500.

In [ ]:
first_model = FeedForwardNN(input_size=784, hidden_size=32, output_size=10)
second_model = FeedForwardNN(input_size=784, hidden_size=250, output_size=10)
third_model = FeedForwardNN(input_size=784, hidden_size=500, output_size=10)

# Función de pérdida

Elegimos la función de pérdida adecuada para nuestro problema de clasificación multiclase.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

# Optimizador

Para hacer más interesante el análisis, realizaremos tres tipos de optimizadores distintos, uno para cada modelo, siendo estos el primero un optimizador ADAM con learning rate de 0.001, el segundo un optimizador SGD con un learning rate de 0.001 y el tercero un optimizador ADAM nuevamente, únicamente que este aumentaremos el learning rate a 0.1.

In [ ]:
import torch.optim as optim

first_optimizer = optim.Adam(first_model.parameters(), lr=0.001)
second_optimizer = optim.SGD(second_model.parameters(), lr=0.001)
third_optimizer = optim.Adam(third_model.parameters(), lr=0.1)

# Entrenamiento

Definimos la función de exactitud.

In [ ]:
def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        logists = model(x)
        predicciones = torch.argmax(logists, dim=1)
        correctas = (predicciones == y).sum().item()
        total = y.shape[0]
    return correctas/total

Definiremos la función de entrenamiento con los parámetros de modelo, función de pérdida, optimizador, número de épocas con valor por defecto de 64, paciencia con valor por defecto de 10 y delta con valor por defecto de 0.005. En esta función mantendremos un historial de la pérdida en el conjunto de validación y lo siguiente será iterar por cada época hasta el número de épocas en el modelo, en modo entrenamiento. Para cada conjunto de batches en Dataloader haremos las predicciones adecuadas y calcularemos la pérdida con nuestra función de pérdida previamente establecida, que es el cross-entropy loss, utilizando las predicciones y la etiqueta con sus valores reales. Restablecemos el optimizador, calculamos el backpropagation de la pérdida y le decimos al optimizador que avance un paso. Cambiamos el modelo a modo de evaluación y ejecutamos las predicciones del modelo con el tensor de validación para obtener los logits correspondientes. Evaluamos la función de cross-entropy loss con estos valores y el tensor de validación para la etiqueta. Calculamos su exactitud y agregamos este valor de pérdida al historial para posteriormente imprimir en pantalla cuál es la época actual, cuál fue la pérdida al entrenar, cuál fue la exactitud al evaluar y cuál fue la pérdida al evaluar. Y de forma condicional implementamos un early stop que, cuando el historial tiene más pasos que el número almacenado por nuestra variable de paciencia: Si la diferencia entre el elemento previo a nuestro rango de paciencia y el último elemento es menor que delta, es decir, si la mejora se ha mantenido mínima o el error ha incrementado (diferencia negativa), entonces se nos acabó la paciencia y terminamos el entrenamiento. Si no, continuamos hasta terminar el número de épocas.

In [ ]:
def train(model, loss_fn, optimizer, num_epochs=64, paciencia=10, delta=0.005):
    historial_eval = []
    for epoch in range(num_epochs):
        model.train()
        for X_batch, y_batch in dataloader:
            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch) # ¿CÓMO QUE PYTHON NO USA BLOCK-LEVEL SCOPING PARA LOS FOR LOOPS?
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            eval_logists = model(x_val_tensor)
            eval_loss = loss_fn(eval_logists, y_val_tensor)
            eval_acc = accuracy(model, x_val_tensor, y_val_tensor)
            historial_eval.append(eval_loss.item())
        
        print(f"Época {epoch+1}")
        print(f"Pérdida al entrenar: {loss.item():.4f}")
        print(f"Exactitud al evaluar: {eval_acc:.4f}")
        print(f"Pérdida al evaluar: {eval_loss:.4f}")
        
        if len(historial_eval) > paciencia and historial_eval[-paciencia-1] - historial_eval[-1] < delta:
            print("Se nos acaba la paciencia.") # yo cuando python
            break

A los tres modelos los entrenaremos con una paciencia de tres, con su respectivo optimizador y la función de pérdida. Sin embargo, al último modelo le permitiremos la friolera de 12 épocas, cuando a los demás les vamos a permitir 64. Veremos cómo se comportan.

In [ ]:
print("Primer modelo")
train(model=first_model, loss_fn=loss_fn, optimizer=first_optimizer, paciencia=3)
print("Segundo modelo")
train(model=second_model, loss_fn=loss_fn, optimizer=second_optimizer, paciencia=3)
print("Tercer modelo")
train(model=third_model, loss_fn=loss_fn, optimizer=third_optimizer, num_epochs=12, paciencia=3)

# Evaluación

Definiremos nuestra función de evaluación con un parámetro, el modelo, al cual estableceremos en modo de evaluación de forma explícita y sin hacer seguimiento del gradiente. Realizaremos las inferencias con nuestro tensor de prueba en el modelo y calculamos la pérdida utilizando el valor real de la observación también del tensor de prueba de la etiqueta. y obtendremos las predicciones en forma de arreglo, de forma que tendremos todos los valores predichos y esto nos servirá para crear nuestra matriz de confusión, a la cual inicializaremos en ceros con la forma de 10x10, ya que tenemos 10 clases, los números del 0 al 9. Y a continuación haremos un stack de nuestras predicciones con nuestra observación real y procederemos a colocar un más uno en cada uno de los elementos correspondientes de nuestra matriz de confusión. Calcularemos la exactitud aprovechando la diagonal y el total. Luego definiremos los vectores de tamaño 10 por las clases en 0 para precisión, recall y F1 score. Luego para cada una de las clases definiremos cuántos verdaderos positivos tenemos, que este es el valor en el índice del igual columnas que de filas en nuestra matriz de confusión. A continuación, calculamos los falsos negativos, que estos son de la fila correspondiente todos los elementos sumados menos los que sí son verdaderos positivos que están en la diagonal. Para los falsos positivos serán todos los elementos de una misma columna sumados menos los verdaderos positivos que son los que están en la diagonal. Finalmente, calculando el recall, será la fórmula que ya conocemos de los verdaderos positivos sobre la suma de verdaderos positivos con falsos negativos. Para precision, son los verdaderos positivos sobre el número de verdaderos positivos más falsos negativos. y el F1 score, pues es el producto de dos por la fracción del producto de precision y recall sobre la suma de precision y recall. Finalmente, nuestra función, una vez iteramos por todas las clases y calculamos estos valores, desplegaremos en pantalla cuál fue la pérdida en prueba, la exactitud en prueba, la matriz de confusión completa, las métricas para cada clase, con el recall, el F1 score y la precisión, y con eso damos por terminada la función de evaluación.

In [ ]:
def eval(model):
    model.eval()
    with torch.no_grad():
        test_logists = model(x_test_tensor)
        test_loss = loss_fn(test_logists, y_test_tensor)
        test_predicciones = torch.argmax(test_logists, dim=1)
        confucio = torch.zeros(10, 10, dtype=torch.int64)
        for tensori, predictori in zip(y_test_tensor, test_predicciones):
            confucio[tensori,predictori] = 1 + confucio[tensori,predictori]
        
        exactitud = torch.diag(confucio).sum().item() / confucio.sum().item()
        
        precisión, recall, f1 = (torch.zeros(10), torch.zeros(10), torch.zeros(10))
        
        for i in range(10):
            atinado = confucio[i,i].item()
            falsos_neg = confucio[i,:].sum().item() - atinado
            falsos_posi = confucio[:,i].sum().item() - atinado
            
            recall[i] = atinado/(atinado+falsos_neg)
            precisión[i] = atinado / (atinado+falsos_posi)
            f1[i] = 2 * (precisión[i]*recall[i])/(precisión[i]+recall[i])
            
        print(f"Pérdida en prueba: {test_loss.item():.4f}")
        print(f"Exactitud en prueba: {exactitud:.4f}")
        print(f"Matriz de confusión")
        print(confucio)
        print("Métricas para cada clase")
        for i in range(10):
            print(f"\tClase {i}")
            print(f"\t\tRecall: {recall[i]:.4f}")
            print(f"\t\tF1: {f1[i]:.4f}")
            print(f"\t\tPrecisión: {precisión[i]:.4f}")
        print("\n")

Ejecutamos la evaluación.

In [ ]:
print("Primer modelo")
eval(model=first_model)
print("Segundo modelo")
eval(model=second_model)
print("Tercer modelo")
eval(model=third_model)